# Лабораторная 7. Генерация лиц (GAN, CelebA)

In [ ]:
!pip install facenet-pytorch scipy -q

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils as vutils
from torchvision.models import inception_v3
from facenet_pytorch import MTCNN
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
from scipy.linalg import sqrtm
import pandas as pd

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

## 1. Подготовка данных

In [ ]:
celeba_img_dir = "data/celeba/img_align_celeba/img_align_celeba"
attr_path = "data/celeba/list_attr_celeba.csv"
crop_dir = "data/celeba_cropped"
IMG_SIZE = 64
N_IMAGES = 20000

attr_df = pd.read_csv(attr_path)
attr_df = attr_df.iloc[:N_IMAGES]
male_labels = ((attr_df["Male"].values + 1) // 2).astype(np.int64)
filenames = attr_df.iloc[:, 0].values
print(f"Изображений: {len(filenames)}, Male: {male_labels.sum()}, Female: {(1 - male_labels).sum()}")

In [ ]:
os.makedirs(crop_dir, exist_ok=True)
mtcnn = MTCNN(image_size=IMG_SIZE, margin=10, post_process=False, device=device)

valid_indices = []
if len([f for f in os.listdir(crop_dir) if f.endswith('.jpg')]) >= N_IMAGES * 0.9:
    print("Кропы уже есть, пропускаем")
    for i in range(N_IMAGES):
        if os.path.exists(os.path.join(crop_dir, f"{i:06d}.jpg")):
            valid_indices.append(i)
else:
    for i in tqdm(range(N_IMAGES), desc="Кроп лиц"):
        img_path = os.path.join(celeba_img_dir, filenames[i])
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert("RGB")
            face = mtcnn(img)
            if face is not None:
                face_pil = transforms.ToPILImage()(face / 255.0)
                face_pil.save(os.path.join(crop_dir, f"{i:06d}.jpg"))
                valid_indices.append(i)
        except Exception:
            continue

labels_array = male_labels[valid_indices]
np.save(os.path.join(crop_dir, "labels.npy"), labels_array)
print(f"Лиц: {len(valid_indices)}")

In [ ]:
class CelebAFaces(Dataset):
    def __init__(self, crop_dir, labels, transform):
        self.crop_dir = crop_dir
        self.labels = labels
        self.transform = transform
        self.files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".jpg")])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.crop_dir, self.files[idx])).convert("RGB")
        img = self.transform(img)
        label = self.labels[idx] if idx < len(self.labels) else 0
        return img, label


transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

labels = np.load(os.path.join(crop_dir, "labels.npy"))
dataset = CelebAFaces(crop_dir, labels, transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
print(f"Dataset: {len(dataset)}, Batches: {len(dataloader)}")

batch, lbls = next(iter(dataloader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow((batch[i] * 0.5 + 0.5).permute(1, 2, 0).numpy())
    ax.set_title("M" if lbls[i] == 1 else "F", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Безусловный DCGAN

DCGAN выбран как стандартная архитектура для генерации 64x64. Generator строится на транспонированных свёртках, Discriminator на обычных.

In [ ]:
nz = 100
ngf = 64
ndf = 64


def weights_init(m):
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1)


netG = Generator().to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)
print(f"G: {sum(p.numel() for p in netG.parameters()):,} params")
print(f"D: {sum(p.numel() for p in netD.parameters()):,} params")

In [ ]:
criterion = nn.BCELoss()
optD = torch.optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optG = torch.optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

history_uncond = {"G": [], "D": []}

for epoch in range(25):
    gl, dl = [], []
    for real, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False):
        bs = real.size(0)
        real = real.to(device)
        ones = torch.ones(bs, device=device)
        zeros = torch.zeros(bs, device=device)

        # D
        netD.zero_grad()
        noise = torch.randn(bs, nz, 1, 1, device=device)
        fake = netG(noise)
        ld = criterion(netD(real), ones) + criterion(netD(fake.detach()), zeros)
        ld.backward()
        optD.step()

        # G
        netG.zero_grad()
        lg = criterion(netD(fake), ones)
        lg.backward()
        optG.step()

        gl.append(lg.item())
        dl.append(ld.item())

    history_uncond["G"].append(np.mean(gl))
    history_uncond["D"].append(np.mean(dl))
    print(f"[{epoch+1}] D={np.mean(dl):.4f} G={np.mean(gl):.4f}")

    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            imgs = netG(fixed_noise).cpu() * 0.5 + 0.5
        grid = vutils.make_grid(imgs[:16], nrow=4, padding=2)
        plt.figure(figsize=(5, 5))
        plt.imshow(grid.permute(1, 2, 0).numpy())
        plt.title(f"Epoch {epoch+1}")
        plt.axis("off")
        plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_uncond["G"], label="G loss")
plt.plot(history_uncond["D"], label="D loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN")
plt.legend()
plt.grid(True)
plt.show()

with torch.no_grad():
    imgs = netG(fixed_noise).cpu() * 0.5 + 0.5
grid = vutils.make_grid(imgs, nrow=8, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("DCGAN — результат")
plt.axis("off")
plt.show()

## 3. Условный GAN (по полу)

Label embedding конкатенируется к шуму в G и к изображению (как доп. канал) в D.

In [ ]:
n_classes = 2


class CGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 10)
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz + 10, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        emb = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([z, emb], dim=1)
        return self.main(x)


class CDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, IMG_SIZE * IMG_SIZE)
        self.main = nn.Sequential(
            nn.Conv2d(4, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_map = self.label_emb(labels).view(-1, 1, IMG_SIZE, IMG_SIZE)
        x = torch.cat([img, label_map], dim=1)
        return self.main(x).view(-1)


cG = CGenerator().to(device)
cD = CDiscriminator().to(device)
cG.apply(weights_init)
cD.apply(weights_init)

In [ ]:
c_optD = torch.optim.Adam(cD.parameters(), lr=0.0002, betas=(0.5, 0.999))
c_optG = torch.optim.Adam(cG.parameters(), lr=0.0002, betas=(0.5, 0.999))
c_fixed_noise = torch.randn(16, nz, 1, 1, device=device)

history_cond = {"G": [], "D": []}

for epoch in range(25):
    gl, dl = [], []
    for real, real_labels in tqdm(dataloader, desc=f"cGAN {epoch+1}", leave=False):
        bs = real.size(0)
        real = real.to(device)
        real_labels = real_labels.to(device)
        ones = torch.ones(bs, device=device)
        zeros = torch.zeros(bs, device=device)

        cD.zero_grad()
        noise = torch.randn(bs, nz, 1, 1, device=device)
        fake_labels = torch.randint(0, n_classes, (bs,), device=device)
        fake = cG(noise, fake_labels)
        ld = criterion(cD(real, real_labels), ones) + criterion(cD(fake.detach(), fake_labels), zeros)
        ld.backward()
        c_optD.step()

        cG.zero_grad()
        lg = criterion(cD(fake, fake_labels), ones)
        lg.backward()
        c_optG.step()

        gl.append(lg.item())
        dl.append(ld.item())

    history_cond["G"].append(np.mean(gl))
    history_cond["D"].append(np.mean(dl))
    print(f"[{epoch+1}] D={np.mean(dl):.4f} G={np.mean(gl):.4f}")

plt.figure(figsize=(10, 4))
plt.plot(history_cond["G"], label="G loss")
plt.plot(history_cond["D"], label="D loss")
plt.title("cGAN")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
labels_f = torch.zeros(16, dtype=torch.long, device=device)
labels_m = torch.ones(16, dtype=torch.long, device=device)

with torch.no_grad():
    fake_f = cG(c_fixed_noise, labels_f).cpu() * 0.5 + 0.5
    fake_m = cG(c_fixed_noise, labels_m).cpu() * 0.5 + 0.5

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(fake_f[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title("F")
    axes[0, i].axis("off")
    axes[1, i].imshow(fake_m[i].permute(1, 2, 0).numpy())
    axes[1, i].set_title("M")
    axes[1, i].axis("off")
plt.suptitle("Один шум — разные метки")
plt.tight_layout()
plt.show()

## 4. WGAN-GP (+5 баллов)

Wasserstein loss + gradient penalty. Critic без Sigmoid, InstanceNorm вместо BatchNorm, n_critic=5.

In [ ]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, IMG_SIZE * IMG_SIZE)
        self.main = nn.Sequential(
            nn.Conv2d(4, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*2, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*4, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*8, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, img, labels):
        label_map = self.label_emb(labels).view(-1, 1, IMG_SIZE, IMG_SIZE)
        x = torch.cat([img, label_map], dim=1)
        return self.main(x).view(-1)


def grad_penalty(critic, real, fake, labels, lam=10.0):
    alpha = torch.rand(real.size(0), 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = critic(interp, labels)
    grads = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    grads = grads.view(grads.size(0), -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean() * lam


wG = CGenerator().to(device)
wC = Critic().to(device)
wG.apply(weights_init)
wC.apply(weights_init)

In [ ]:
w_optC = torch.optim.Adam(wC.parameters(), lr=0.0001, betas=(0.0, 0.9))
w_optG = torch.optim.Adam(wG.parameters(), lr=0.0001, betas=(0.0, 0.9))
n_critic = 5
w_fixed = torch.randn(16, nz, 1, 1, device=device)

history_wgan = {"G": [], "C": [], "wd": []}

for epoch in range(25):
    gl, cl, wds = [], [], []
    step = 0
    for real, real_labels in tqdm(dataloader, desc=f"WGAN {epoch+1}", leave=False):
        bs = real.size(0)
        real = real.to(device)
        real_labels = real_labels.to(device)

        wC.zero_grad()
        noise = torch.randn(bs, nz, 1, 1, device=device)
        fl = torch.randint(0, n_classes, (bs,), device=device)
        fake = wG(noise, fl).detach()
        sr = wC(real, real_labels).mean()
        sf = wC(fake, fl).mean()
        gp = grad_penalty(wC, real, fake, real_labels)
        lc = sf - sr + gp
        lc.backward()
        w_optC.step()
        cl.append(lc.item())
        wds.append((sr - sf).item())
        step += 1

        if step % n_critic == 0:
            wG.zero_grad()
            noise = torch.randn(bs, nz, 1, 1, device=device)
            fl = torch.randint(0, n_classes, (bs,), device=device)
            fake = wG(noise, fl)
            lg = -wC(fake, fl).mean()
            lg.backward()
            w_optG.step()
            gl.append(lg.item())

    history_wgan["G"].append(np.mean(gl) if gl else 0)
    history_wgan["C"].append(np.mean(cl))
    history_wgan["wd"].append(np.mean(wds))
    print(f"[{epoch+1}] C={np.mean(cl):.4f} G={np.mean(gl) if gl else 0:.4f} WD={np.mean(wds):.4f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history_wgan["G"], label="G")
a1.plot(history_wgan["C"], label="C")
a1.set_title("WGAN-GP Loss")
a1.legend(); a1.grid(True)
a2.plot(history_wgan["wd"], color="green")
a2.set_title("Wasserstein Distance")
a2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    wf = wG(w_fixed, labels_f).cpu() * 0.5 + 0.5
    wm = wG(w_fixed, labels_m).cpu() * 0.5 + 0.5

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(wf[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[0, i].set_title("F"); axes[0, i].axis("off")
    axes[1, i].imshow(wm[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].set_title("M"); axes[1, i].axis("off")
plt.suptitle("WGAN-GP: один шум — разные метки")
plt.tight_layout()
plt.show()

## 5. FID и Inception Score

In [ ]:
inception = inception_v3(pretrained=True, transform_input=False).to(device)
inception.eval()

features_cache = {}
def hook(module, inp, out):
    features_cache["f"] = out
inception.avgpool.register_forward_hook(hook)


def get_features(imgs):
    imgs = F.interpolate(imgs, size=(299, 299), mode="bilinear", align_corners=False).to(device)
    with torch.no_grad():
        inception(imgs)
    return features_cache["f"].view(imgs.size(0), -1).cpu().numpy()


def get_probs(imgs):
    imgs = F.interpolate(imgs, size=(299, 299), mode="bilinear", align_corners=False).to(device)
    with torch.no_grad():
        logits = inception(imgs)
    return F.softmax(logits, dim=1).cpu().numpy()


def calc_fid(real_f, fake_f):
    mu_r, mu_f = real_f.mean(0), fake_f.mean(0)
    s_r = np.cov(real_f, rowvar=False)
    s_f = np.cov(fake_f, rowvar=False)
    diff = mu_r - mu_f
    covmean = sqrtm(s_r @ s_f)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(s_r + s_f - 2 * covmean))


def calc_is(probs, splits=5):
    scores = []
    n = len(probs)
    cs = n // splits
    for i in range(splits):
        chunk = probs[i*cs:(i+1)*cs]
        py = chunk.mean(0, keepdims=True)
        kl = (chunk * (np.log(chunk + 1e-10) - np.log(py + 1e-10))).sum(1).mean()
        scores.append(np.exp(kl))
    return float(np.mean(scores)), float(np.std(scores))

In [ ]:
n_eval = 1000

real_feats = []
cnt = 0
for imgs, _ in dataloader:
    real_feats.append(get_features(imgs * 0.5 + 0.5))
    cnt += imgs.size(0)
    if cnt >= n_eval:
        break
real_feats = np.concatenate(real_feats)[:n_eval]


def eval_gen(gen, name, conditional=False):
    ff, pp = [], []
    for i in range(0, n_eval, 50):
        bs = min(50, n_eval - i)
        noise = torch.randn(bs, nz, 1, 1, device=device)
        with torch.no_grad():
            if conditional:
                lbl = torch.randint(0, 2, (bs,), device=device)
                fake = gen(noise, lbl) * 0.5 + 0.5
            else:
                fake = gen(noise) * 0.5 + 0.5
        ff.append(get_features(fake))
        pp.append(get_probs(fake))
    ff = np.concatenate(ff)
    pp = np.concatenate(pp)
    fid = calc_fid(real_feats, ff)
    is_m, is_s = calc_is(pp)
    print(f"{name}: FID={fid:.1f}, IS={is_m:.2f}±{is_s:.2f}")
    return fid, is_m, is_s


fid1, is1m, is1s = eval_gen(netG, "DCGAN")
fid2, is2m, is2s = eval_gen(cG, "cGAN", conditional=True)
fid3, is3m, is3s = eval_gen(wG, "WGAN-GP", conditional=True)

print()
print(f"{'Model':<12} {'FID':>8} {'IS':>12}")
print("-" * 35)
print(f"{'DCGAN':<12} {fid1:>8.1f} {is1m:>6.2f}±{is1s:.2f}")
print(f"{'cGAN':<12} {fid2:>8.1f} {is2m:>6.2f}±{is2s:.2f}")
print(f"{'WGAN-GP':<12} {fid3:>8.1f} {is3m:>6.2f}±{is3s:.2f}")

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
noise_cmp = torch.randn(8, nz, 1, 1, device=device)
lbl_cmp = torch.zeros(8, dtype=torch.long, device=device)

with torch.no_grad():
    i1 = netG(noise_cmp).cpu() * 0.5 + 0.5
    i2 = cG(noise_cmp, lbl_cmp).cpu() * 0.5 + 0.5
    i3 = wG(noise_cmp, lbl_cmp).cpu() * 0.5 + 0.5

for row, (imgs, title) in enumerate([(i1, "DCGAN"), (i2, "cGAN"), (i3, "WGAN-GP")]):
    for col in range(8):
        axes[row, col].imshow(imgs[col].permute(1, 2, 0).clamp(0, 1).numpy())
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(title, fontsize=11, rotation=0, labelpad=50)

plt.suptitle("Сравнение моделей")
plt.tight_layout()
plt.show()

## Выводы

- DCGAN генерирует лица, но без контроля атрибутов
- cGAN управляет генерацией через метку пола: один шум + разные метки = разные лица
- WGAN-GP стабильнее обучается (Wasserstein distance растёт, кривые глаже)
- FID ниже = лучше, IS выше = лучше
- Предобработка лиц через MTCNN даёт выравнивание и единый размер